# 03 – Feature Engineering

**Project**: DengAI – Predicting Disease Spread  

---

### Objective
Build predictive features informed by dengue biology:
- **Temporal lags** (mosquito lifecycle ~2 weeks; transmission delay ~1–2 weeks)
- **Rolling windows** to capture environment trends
- **Domain features**: temperature suitability, humidity-temperature interaction
- **Cyclical encoding** of week/month

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

ROOT = Path('../')
PROC = ROOT / 'data/processed'

train = pd.read_csv(PROC / 'train_clean.csv', parse_dates=['week_start_date'])
test  = pd.read_csv(PROC / 'test_clean.csv',  parse_dates=['week_start_date'])
print("Loaded:", train.shape, test.shape)

Loaded: (1456, 25) (416, 24)


In [2]:
KEY_VARS = [
    'reanalysis_specific_humidity_g_per_kg',
    'reanalysis_dew_point_temp_k',
    'station_avg_temp_c',
    'station_min_temp_c',
    'precipitation_amt_mm',
    'reanalysis_relative_humidity_percent',
]

def engineer_features(cdf):
    cdf = cdf.copy().sort_values('week_start_date').reset_index(drop=True)

    # Cyclical time encoding
    cdf['sin_week'] = np.sin(2 * np.pi * cdf['weekofyear'] / 52)
    cdf['cos_week'] = np.cos(2 * np.pi * cdf['weekofyear'] / 52)
    cdf['month']    = cdf['week_start_date'].dt.month
    cdf['quarter']  = cdf['week_start_date'].dt.quarter

    # Lag features: 1–4 weeks (short-term), 8–12 weeks (incubation + mosquito lifecycle)
    for var in KEY_VARS:
        for lag in [1, 2, 3, 4, 8, 12]:
            cdf[f'{var}_lag{lag}'] = cdf[var].shift(lag)

    # Rolling means (shifted by 1 to avoid leakage)
    for var in KEY_VARS:
        for win in [4, 8, 12]:
            cdf[f'{var}_roll{win}'] = cdf[var].shift(1).rolling(win).mean()

    # Domain features
    cdf['temp_suitability'] = cdf['station_avg_temp_c'].apply(
        lambda t: max(0, 1 - abs(t - 27.5) / 15) if pd.notnull(t) else np.nan
    )
    cdf['temp_range']   = cdf['station_max_temp_c'] - cdf['station_min_temp_c']
    cdf['humid_temp']   = cdf['reanalysis_relative_humidity_percent'] * cdf['station_avg_temp_c']
    cdf['ndvi_mean']    = cdf[[c for c in cdf.columns if 'ndvi' in c and '_lag' not in c
                                 and '_roll' not in c]].mean(axis=1)
    cdf['precip_roll4'] = cdf['precipitation_amt_mm'].shift(1).rolling(4).sum()
    cdf['precip_roll8'] = cdf['precipitation_amt_mm'].shift(1).rolling(8).sum()

    # City flag
    cdf['is_sj'] = (cdf['city'] == 'sj').astype(int)
    return cdf

train_fe = engineer_features(train)
test_fe  = engineer_features(test)

# Backfill NaNs introduced by lags at start of series
feat_cols = [c for c in train_fe.columns if c not in
             ['city','week_start_date','total_cases']]
train_fe[feat_cols] = train_fe[feat_cols].bfill()
test_fe[feat_cols]  = test_fe[feat_cols].bfill()

print(f"Features before engineering: {train.shape[1] - 3}")
print(f"Features after engineering:  {len(feat_cols)}")

Features before engineering: 22
Features after engineering:  87


In [3]:
# Save
train_fe.to_csv(PROC / 'train_features.csv', index=False)
test_fe.to_csv(PROC  / 'test_features.csv',  index=False)
print("Saved train_features.csv and test_features.csv")
print("Sample feature names:", feat_cols[:8])

Saved train_features.csv and test_features.csv
Sample feature names: ['year', 'weekofyear', 'ndvi_ne', 'ndvi_nw', 'ndvi_se', 'ndvi_sw', 'precipitation_amt_mm', 'reanalysis_air_temp_k']


## Feature Engineering Summary

| Category | Features | Count |
|----------|----------|-------|
| Original weather | Raw sensor/reanalysis cols | 20 |
| Temporal lags (1,2,3,4,8,12 wks) | 6 key vars × 6 lags | 36 |
| Rolling means (4,8,12 wk windows) | 6 key vars × 3 windows | 18 |
| Cyclical time | sin_week, cos_week, month, quarter | 4 |
| Domain features | temp_suitability, humid_temp, temp_range, ndvi_mean, precip_roll4/8 | 6 |
| City flag | is_sj | 1 |
| **Total** | | **87** |

**Key design decisions:**
- Lags shifted by city to avoid contamination between San Juan and Iquitos series
- Rolling windows shifted by 1 before computing to prevent target leakage
- Temperature suitability peaks at 27.5°C (optimal for *Aedes aegypti* breeding)